## 1) Importar bibliotecas

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.pipeline import Pipeline
from matplotlib.axes import Axes
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, QuantileTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import PredictionErrorDisplay

## 2) Base de dados

### 2.1) Leitura

In [ ]:
dataset = pl.read_parquet(
    source = "./diabetes_dataset.parquet"
)

print(dataset.shape)
dataset.head(2)

### 2.2) Análise de outliers

#### 2.2.1) Função de boxplots

In [ ]:
def plot_boxplots(
    ax: Axes,
    dataframe: pl.DataFrame,
    title: str,
) -> None:

    sns.boxplot(
        data = dataframe,
        ax = ax,
    )

    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks(ticks = np.arange(dataframe.columns.__len__()), labels = dataframe.columns)
    ax.set_title(
        label = title, fontsize = 20, fontname = "arial"
    )

In [ ]:
columns = [
    "age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"
]

fig, ax = plt.subplots(
    figsize = (12, 5)
)

plot_boxplots(
    ax = ax,
    dataframe = dataset[columns],
    title = "Features outliers"
)

plt.show()

#### 2.2.2) Função de histogramas

<p>Como podemos ver, apenas 2 colunas chamam a atenção com relação a sua assimetria:</p>

<ul>
    <li>s4</li>
    <li>s3</li>
</ul>

<p>Ambas, além de possuir outliers, têm em outras palavras uma assimetria maior que as demais colunas numéricas. Portanto, para a transformação, será necessário outro processamento chamado <b>PowerTransformer</b>, diferentemente do padrão já esperado em <b>StandardScaler</b>.</p>

In [ ]:
def plot_boxplots(
    ax: Axes,
    dataframe: pl.DataFrame,
    title: str,
    bins: str = "sturges",
) -> None:
    
    sns.histplot(
        data = dataframe,
        ax = ax,
        kde = True,
        bins = bins
    )

    ax.spines[["top", "right"]].set_visible(False)
    
    ax.set_title(
        label = title, fontsize = 16, fontname = "arial"
    )

In [ ]:
columns = [
    "age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6", "target",
]

fig, axs = plt.subplots(
    figsize = (16, 8),
    ncols = 5,
    nrows = 2,
    gridspec_kw = {
        "wspace": 0.2,
        "hspace": 0.5
    }
)

axs = axs.flatten()

for index, column in enumerate(columns):

    plot_boxplots(
        ax = axs[index],
        dataframe = dataset[column],
        title = f"Column {column}\nSkew: {dataset[column].skew():.2f}"
    )

plt.suptitle("Features distributions", fontsize = 22, )
plt.show()

## 3) Create pipeline

### 3.1) Define transformers

#### 3.1.1) StandardScaler

In [ ]:
standard_scaler = StandardScaler()
columns_standard_scaler = ["age", "bmi", "bp", "s1", "s2", "s5", "s6"]

#### 3.1.2) PowerTransformer

In [ ]:
power_transformer = PowerTransformer(method = "box-cox")
columns_power_transformer = ["s3", "s4"]

#### 3.1.3) OneHotEncoder

In [ ]:
one_hot_encoder = OneHotEncoder(
    drop = "if_binary",
)
columns_one_hot_encoder = ["sex"]

### 3.2) Transformer

In [ ]:
transformer = ColumnTransformer(
    transformers = [
        ("standard_scaler", standard_scaler, columns_standard_scaler),
        ("power_transformer", power_transformer, columns_power_transformer),
        ("one_hot_encoder", one_hot_encoder, columns_one_hot_encoder)
    ],
    remainder = 'passthrough',
)

### 3.2) Setting pipeline

In [ ]:
pipeline = Pipeline(
    steps = [
        ("preprocessor", transformer),
        ("linear_regression", LinearRegression())
    ]
)

pipeline

### 3.3) TransformedTargetRegressor

In [ ]:
regressor_transformed_target = TransformedTargetRegressor(
    regressor = pipeline,
    transformer = QuantileTransformer(
        n_quantiles = 20, output_distribution = "normal"
    )
)

regressor_transformed_target

## 4) Trainning dataset

### 4.1) Splitting dataset to train and test

In [ ]:
X = dataset.drop("target")
y = dataset["target"]

### 4.2) Fitting pipeline

In [ ]:
kf = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 1
)

scores = cross_validate(
    estimator = regressor_transformed_target,
    X = X,
    y = y,
    cv = kf,
    scoring = ["r2", "neg_mean_squared_error", "neg_mean_absolute_error"]
)

scores = pl.DataFrame(
    scores
)

display(scores)